# Prove that the correctness of a question is independent of the model used / same model multiple sampling rounds

## Load libraries and load data

In [1]:
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif, chi2, mutual_info_classif
import numpy as np
import pandas as pd
from collections import Counter
import math
import utils
import json
import os
from utils import postprocess_responses
from collections import defaultdict
from collections import Counter

np.random.seed(42)

c:\Users\nanfangwuyu\.conda\envs\RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# MODEL_LIST = ["qwen_7b"]

# settings = ['cot_trigger']


# settings = ['ncot_force', 'ncot_force', 'ncot_force', ]

# MODEL_LIST = ["qwen_3b", "qwen_7b", "qwen_32b", "qwen_72b"]

# settings = ['cot_trigger', 'cot_trigger', 'cot_trigger', 'cot_trigger',]
# settings = ['ncot_force', 'ncot_force', 'ncot_force', 'ncot_force',]



MODEL_LIST = ["llama", "pixtral", "gemma3_12b", "qwen_3b", "qwen_7b", "qwen_32b", "qwen_72b"]
settings = ['cot_trigger', 'cot_trigger', 'cot_trigger', 'cot_trigger', 'cot_trigger', 'cot_trigger', 'cot_trigger']
DATASET_LIST = ["MathVista", "MathVision", "TQA", "ScienceQA", "MMStar", "MMMU"]

N = 16

In [3]:
def calculate_entropy_distribution(ans_dist, n):
    
    def cal_entropy(answer_list):
        # -∑(p_i * log(p_i))

        counts = defaultdict(int)
        for answer in answer_list:
            if 0 <= answer <= 9:  # Assuming possible answers are in the range 0-9
                counts[answer] += 1
            else:
                print(f"Error! Idx {answer}")

        total_answers = len(answer_list)

        entropy = 0.0
        for count in counts.values():
            if count > 0:
                probability = count / total_answers
                entropy -= probability * math.log2(probability)

        max_count = max(counts.values())
        max_keys = [k for k, v in counts.items() if v == max_count]
        best_choice = np.random.choice(max_keys)

        return entropy, best_choice

    def cal_entropy_scipy(answer_list, len_ops):
        from scipy.stats import entropy

        # count for i in range (0, len_ops), each appears how many times
        counts = {i: np.sum(answer_list == i) for i in range(len_ops)}
        pk = np.array(list(counts.values()), dtype=float)
        # pk = [9,1,0,0,0]
        # print(pk)
        n = len_ops  # number of discrete categories
        H = entropy(pk, base=2)           
        # print(H)   # entropy in bits
        H_norm = H / (np.log2(n))              # normalized to max entropy of log2(n)

        max_count = max(counts.values())
        max_keys = [k for k, v in counts.items() if v == max_count]
        best_choice = np.random.choice(max_keys)
        return H_norm, best_choice

    entropy_distribution = []
    choice_list = []
    for j in range(len(ans_dist)):
        ans_dist_j = ans_dist[j][:n]
        # len_ops = len(ds[option_s][j])
        # entropy, best_choice = cal_entropy(ans_dist_j)
        entropy, best_choice = cal_entropy_scipy(ans_dist_j, len_ops=option_lengths[j])
        entropy_distribution.append(entropy)
        choice_list.append(best_choice)
    return entropy_distribution, choice_list

def calculate_majority_distribution(ans_dist, n):
    
    def cal_majority(answer_list):

        counts = defaultdict(int)
        for answer in answer_list:
            if 0 <= answer <= 9:
                counts[answer] += 1
            else:
                print(f"Error! Idx {answer}")

        # Find the majority choice (the most frequent answer)
        max_count = max(counts.values())
        max_keys = [k for k, v in counts.items() if v == max_count]
        majority_choice = np.random.choice(max_keys)
        return majority_choice

    choice_list = []
    for j in range(len(ans_dist)):
        ans_dist_j = ans_dist[j][:n]
        majority_choice = cal_majority(ans_dist_j)
        # If there is at least one correct answer
        choice_list.append(majority_choice)
    return choice_list


def calculate_accuracy(choice_list, idx_ground_truth):
    correct_count = sum(1 for i, choice in enumerate(choice_list) if choice == idx_ground_truth[i])
    accuracy = correct_count / len(idx_ground_truth)
    return accuracy

def calculate_match(choice_list, idx_ground_truth):
    match_list = [1 if choice == idx_ground_truth[i] else 0 for i, choice in enumerate(choice_list)]
    return match_list

## Correlation Coefficient

In [4]:
def average_pairwise_nmi(preds, labels=None):
    """
    preds: list of length U, each element is array shape (n_items,) of predictions
    labels: not needed for NMI, just predictions
    """
    from sklearn.metrics import normalized_mutual_info_score
    U = len(preds)
    nmi_vals = []
    for i in range(U):
        for j in range(i+1, U):
            nmi_vals.append(normalized_mutual_info_score(
                preds[i], preds[j], average_method='min'))
    return 2 * np.sum(nmi_vals) / (U * (U - 1))

def average_pairwise_rho(preds, labels):
    """
    preds: list of length U, each element is array shape (n_items,) of predictions
    labels: array shape (n_items,) of true labels
    """
    U = len(preds)
    rho_vals = []
    for i in range(U):
        for j in range(i+1, U):
            Z_i = (preds[i] == labels).astype(int)
            Z_j = (preds[j] == labels).astype(int)
            p = Z_i.mean()
            cov = np.mean(Z_i * Z_j) - p**2
            rho_vals.append(cov / (p * (1 - p)))
    return 2 * np.sum(rho_vals) / (U * (U - 1))


In [57]:
average_pairwise_rho(multi_match_list_all[5], idx_ground_truth)

np.float64(0.7397389017794682)

In [5]:

nmi_all = defaultdict(dict)
rho_all = defaultdict(dict)
nmi_qwen7b = defaultdict(str)
rho_qwen7b = defaultdict(str)
for DATASET_NAME in DATASET_LIST:
    ds = utils.load_dataset_(DATASET_NAME)
    option_s = 'options' if 'options' in ds[0] else 'choices'

    multi_cot_responses = []
    multi_file_names = []
    metas = []
    for i, (MODEL_NAME, setting) in enumerate(zip(MODEL_LIST, settings)):
        metas.append((MODEL_NAME, setting))
        cot_responses_list, file_names = utils.load_latest_response(f'data/n_vs_one/{DATASET_NAME}/{MODEL_NAME}/responses_{MODEL_NAME}_{DATASET_NAME}_{setting}', N, debug=False)
        cot_responses = [utils.clean_responses(item, mode=MODEL_NAME, setting=setting) for item in cot_responses_list]
        file_names = [file_name.split('_')[-1].split('.')[0] for file_name in file_names]

        multi_cot_responses.append(cot_responses)
        multi_file_names.append(file_names)
    print(N, min(len(x) for x in multi_file_names))
    assert N == min(len(x) for x in multi_file_names)
    assert len(ds) == len(multi_cot_responses[0][0])

    multi_lst_AE_all, multi_match_list_all = [], []
    for i, meta in enumerate(metas):
        MODEL_NAME, setting = meta
        lst_AE_all, match_list_all = postprocess_responses(ds, DATASET_NAME, MODEL_NAME, multi_cot_responses[i], multi_file_names[i])
        multi_lst_AE_all.append(lst_AE_all)
        multi_match_list_all.append(match_list_all)

    idx_ground_truth = []
    for i, ans in enumerate(ds['answer']):
        if DATASET_NAME in ["MathVista"]:
            idx_ground_truth.append(ds[option_s][i].index(ans))
        elif DATASET_NAME in ["ScienceQA", "TQA"]:
            idx_ground_truth.append(ans)
        else:
            chara_list = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
            idx_ground_truth.append(chara_list.index(ans))

    
    for i, (MODEL_NAME, setting) in enumerate(metas):
        nmi_all[MODEL_NAME][DATASET_NAME] = average_pairwise_nmi(np.array(multi_match_list_all[i]), idx_ground_truth)
        rho_all[MODEL_NAME][DATASET_NAME] = average_pairwise_rho(np.array(multi_match_list_all[i]), idx_ground_truth)

        if MODEL_NAME == 'qwen_7b':
            nmi_ = ""
            rho_ = ""
            for n in range(2, 16+1):
                nmi_ += (f"            ({n}, {average_pairwise_nmi(np.array(multi_match_list_all[0][:n]), idx_ground_truth)})\n")
            for n in range(2, 16+1):
                rho_ += (f"            ({n}, {average_pairwise_rho(np.array(multi_match_list_all[0][:n]), idx_ground_truth)})\n")
            nmi_qwen7b[DATASET_NAME] = nmi_
            rho_qwen7b[DATASET_NAME] = rho_


16 16
Processing llama 075345 ...
Processing llama 075215 ...
Processing llama 022037 ...
Processing llama 075059 ...
Processing llama 074709 ...
Processing llama 060530 ...
Processing llama 060521 ...
Processing llama 060337 ...
Processing llama 060108 ...
Processing llama 041507 ...
Processing llama 041418 ...
Processing llama 041401 ...
Processing llama 041012 ...
Processing llama 022735 ...
Processing llama 022353 ...
Processing llama 021651 ...
Processing pixtral 022022 ...
Processing pixtral 035635 ...
Processing pixtral 070958 ...
Processing pixtral 053321 ...
Processing pixtral 202052 ...
Processing pixtral 201045 ...
Processing pixtral 201630 ...
Processing pixtral 183922 ...
Processing pixtral 184249 ...
Processing pixtral 183641 ...
Processing pixtral 170530 ...
Processing pixtral 170201 ...
Processing pixtral 152659 ...
Processing pixtral 170135 ...
Processing pixtral 152712 ...
Processing pixtral 152310 ...
Processing gemma3_12b 032407 ...
Processing gemma3_12b 032158 ...


In [6]:
# mv_delta_all = dict()
setting = 'cot_trigger'

import pandas as pd

def cal_multi_round_voting(n):
    accs_list = []
    for i in range(len(metas)):
        accs = multi_major_acc_all[i][:n]
        accs_list.append(np.mean(accs))
    return accs_list

df = pd.DataFrame(columns=MODEL_LIST, index=DATASET_LIST)
for DATASET_NAME in DATASET_LIST:
    ds = utils.load_dataset_(DATASET_NAME)
    option_s = 'options' if 'options' in ds[0] else 'choices'
    N = 16
    np.random.seed(42)
    # force_reprocess = True
    force_reprocess = False
    multi_cot_responses = []
    multi_file_names = []
    metas = []
    for MODEL_NAME, setting in zip(MODEL_LIST, settings):
        metas.append((MODEL_NAME, setting))
        cot_responses_list, file_names = utils.load_latest_response(f'data/n_vs_one/{DATASET_NAME}/{MODEL_NAME}/responses_{MODEL_NAME}_{DATASET_NAME}_{setting}', N, debug=False)
        cot_responses = [utils.clean_responses(item, mode=MODEL_NAME, setting=setting) for item in cot_responses_list]
        file_names = [file_name.split('_')[-1].split('.')[0] for file_name in file_names]

        multi_cot_responses.append(cot_responses)
        multi_file_names.append(file_names)
    print(N, min(len(x) for x in multi_file_names))
    assert N == min(len(x) for x in multi_file_names)
    assert len(ds) == len(multi_cot_responses[0][0])
    multi_lst_AE_all, multi_match_list_all = [], []
    for i, meta in enumerate(metas):
        MODEL_NAME, setting = meta
        lst_AE_all, match_list_all = postprocess_responses(ds, DATASET_NAME, MODEL_NAME, multi_cot_responses[i], multi_file_names[i], force_reprocess=force_reprocess)
        multi_lst_AE_all.append(lst_AE_all)
        multi_match_list_all.append(match_list_all)
    option_lengths = [len(ops) for ops in ds[option_s]]
    idx_ground_truth = []
    for i, ans in enumerate(ds['answer']):
        if DATASET_NAME in ["MathVista"]:
            idx_ground_truth.append(ds[option_s][i].index(ans))
        elif DATASET_NAME in ["ScienceQA", "TQA"]:
            idx_ground_truth.append(ans)
        else:
            chara_list = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
            idx_ground_truth.append(chara_list.index(ans))
    multi_lst_AE_all = np.array(multi_lst_AE_all)
    multi_ent_dist_all = []
    multi_ent_choice_all = []
    multi_ent_acc_all = []
    multi_ent_match_all = []
    for ans_dist in multi_lst_AE_all:
        ent_dist_all = []
        ent_choice_all = []
        ent_acc_all = []
        ent_match_all = []
        for n in range(1, N + 1):
            entropy_distribution, entropy_choices = calculate_entropy_distribution(ans_dist.T, n)
            ent_dist_all.append(entropy_distribution)
            ent_choice_all.append(entropy_choices)
            ent_acc_all.append(calculate_accuracy(entropy_choices, idx_ground_truth))
            ent_match_all.append(calculate_match(entropy_choices, idx_ground_truth))
        multi_ent_dist_all.append(ent_dist_all)
        multi_ent_choice_all.append(ent_choice_all)
        multi_ent_acc_all.append(ent_acc_all)
        multi_ent_match_all.append(ent_match_all)

    multi_major_choice_all = []
    multi_major_acc_all = []

    for i, ans_dist in enumerate(multi_lst_AE_all):
        major_choice_all = []
        major_acc_all = []
        for n in range(1, N + 1):
            major_choices = calculate_majority_distribution(ans_dist.T, n)
            major_choice_all.append(major_choices)
            major_acc_all.append(calculate_accuracy(major_choices, idx_ground_truth))
        multi_major_choice_all.append(major_choice_all)
        multi_major_acc_all.append(major_acc_all)
        
    multi_cot_responses = np.array(multi_cot_responses, dtype=object)  # shape: (3, 16, 540)
    multi_lst_AE_all = np.array(multi_lst_AE_all)  # shape: (3, 16, 540)
    multi_ent_dist_all = np.array(multi_ent_dist_all)  # shape: (3, 16, 540)
    multi_ent_choice_all = np.array(multi_ent_choice_all)  # shape: (3, 16, 540)
    multi_ent_acc_all = np.array(multi_ent_acc_all) # shape: (3, 16)
    multi_ent_match_all = np.array(multi_ent_match_all)  # shape: (3, 16, 540)
    multi_major_choice_all = np.array(multi_major_choice_all)  # shape: (3, 16, 540)
    multi_major_acc_all = np.array(multi_major_acc_all)  # shape: (3, 16)
    multi_match_list_all = np.array(multi_match_list_all)  # shape: (3, 16, 540)

    def pipe(n=N):
        results = {'multi_round': {}, 'multi_model': {}}
        # 多轮投票
        # results['multi_round']['avg_acc'] = cal_multi_round_avg_acc(n)
        # results['multi_round']['pivot_acc'] = cal_pivot_acc(n)
        results['multi_round']['voting_acc'] = cal_multi_round_voting(n)
        # results['multi_round']['length'] = cal_multi_round_length(n)

        return results

    # 运行管道
    results = pipe()
    print("\nFinal Results:")
    for key, value in results.items():
        for sub_key, sub_value in value.items():
            print(f"{key} - {sub_key}: {sub_value}")

    table = []
    di = dict(zip(MODEL_LIST, results['multi_round']['voting_acc']))
    df.loc[DATASET_NAME] = di
    display(df)
df.loc['average'] = df.mean()
df.to_csv(f"data/final_results/multi_round/voting_{MODEL_LIST}.csv", index=True)



16 16
Processing llama 075345 ...
Processing llama 075215 ...
Processing llama 022037 ...
Processing llama 075059 ...
Processing llama 074709 ...
Processing llama 060530 ...
Processing llama 060521 ...
Processing llama 060337 ...
Processing llama 060108 ...
Processing llama 041507 ...
Processing llama 041418 ...
Processing llama 041401 ...
Processing llama 041012 ...
Processing llama 022735 ...
Processing llama 022353 ...
Processing llama 021651 ...
Processing pixtral 022022 ...
Processing pixtral 035635 ...
Processing pixtral 070958 ...
Processing pixtral 053321 ...
Processing pixtral 202052 ...
Processing pixtral 201045 ...
Processing pixtral 201630 ...
Processing pixtral 183922 ...
Processing pixtral 184249 ...
Processing pixtral 183641 ...
Processing pixtral 170530 ...
Processing pixtral 170201 ...
Processing pixtral 152659 ...
Processing pixtral 170135 ...
Processing pixtral 152712 ...
Processing pixtral 152310 ...
Processing gemma3_12b 032407 ...
Processing gemma3_12b 032158 ...


,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.548843,0.601968,0.683681,0.599769,0.772801,0.822917,0.833218
MathVision,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TQA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ScienceQA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MMStar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MMMU,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Warning, you may use the wrong setting
16 16
Processing llama 023856 ...
Processing llama 023743 ...
Processing llama 023531 ...
Processing llama 023441 ...
Processing llama 023102 ...
Processing llama 022726 ...
Processing llama 022450 ...
Processing llama 022533 ...
Processing llama 022100 ...
Processing llama 021934 ...
Processing llama 021433 ...
Processing llama 020755 ...
Processing llama 021426 ...
Processing llama 021422 ...
Processing llama 021301 ...
Processing llama 021149 ...
Processing pixtral 080402 ...
Processing pixtral 112147 ...
Processing pixtral 112550 ...
Processing pixtral 112748 ...
Processing pixtral 112804 ...
Processing pixtral 112831 ...
Processing pixtral 112939 ...
Processing pixtral 113028 ...
Processing pixtral 113116 ...
Processing pixtral 113400 ...
Processing pixtral 114236 ...
Processing pixtral 120114 ...
Processing pixtral 152337 ...
Processing pixtral 224509 ...
Processing pixtral 155331 ...
Processing pixtral 154725 ...
Processing gemma3_12b 23140

,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.548843,0.601968,0.683681,0.599769,0.772801,0.822917,0.833218
MathVision,0.257588,0.273866,0.331511,0.236904,0.334122,0.415919,0.457001
TQA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ScienceQA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MMStar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MMMU,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Warning, you may use the wrong setting
Warning, you may use the wrong setting
16 16
Processing llama 102140 ...
Processing llama 101645 ...
Processing llama 101114 ...
Processing llama 101113 ...
Processing llama 072606 ...
Processing llama 071312 ...
Processing llama 023728 ...
Processing llama 215039 ...
Processing llama 022824 ...
Processing llama 214540 ...
Processing llama 170602 ...
Processing llama 170151 ...
Processing llama 121938 ...
Processing llama 121522 ...
Processing llama 072827 ...
Processing llama 072927 ...
Processing pixtral 134456 ...
Processing pixtral 134335 ...
Processing pixtral 134132 ...
Processing pixtral 133807 ...
Processing pixtral 133852 ...
Processing pixtral 133702 ...
Processing pixtral 071213 ...
Processing pixtral 133623 ...
Processing pixtral 133530 ...
Processing pixtral 133407 ...
Processing pixtral 133402 ...
Processing pixtral 133253 ...
Processing pixtral 132946 ...
Processing pixtral 110926 ...
Processing pixtral 110548 ...
Processing pixtral

...

,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.548843,0.601968,0.683681,0.599769,0.772801,0.822917,0.833218
MathVision,0.257588,0.273866,0.331511,0.236904,0.334122,0.415919,0.457001
TQA,0.759932,0.805365,0.802968,0.690677,0.810674,0.847489,0.858219
ScienceQA,0.829853,0.815754,0.785635,0.726543,0.816404,0.851326,0.854766
MMStar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MMMU,NaN,NaN,NaN,NaN,NaN,NaN,NaN


16 16
Processing llama 062727 ...
Processing llama 063048 ...
Processing llama 094231 ...
Processing llama 094832 ...
Processing llama 125741 ...
Processing llama 130723 ...
Processing llama 161236 ...
Processing llama 162344 ...
Processing llama 192924 ...
Processing llama 194100 ...
Processing llama 224552 ...
Processing llama 225730 ...
Processing llama 021651 ...
Processing llama 020204 ...
Processing llama 051652 ...
Processing llama 053254 ...
Processing pixtral 054748 ...
Processing pixtral 054815 ...
Processing pixtral 082105 ...
Processing pixtral 082344 ...
Processing pixtral 105411 ...
Processing pixtral 105737 ...
Processing pixtral 132949 ...
Processing pixtral 133448 ...
Processing pixtral 160451 ...
Processing pixtral 161248 ...
Processing pixtral 184031 ...
Processing pixtral 184953 ...
Processing pixtral 212058 ...
Processing pixtral 212616 ...
Processing pixtral 235651 ...
Processing pixtral 000029 ...
Processing gemma3_12b 050249 ...
Processing gemma3_12b 050615 ...


,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.548843,0.601968,0.683681,0.599769,0.772801,0.822917,0.833218
MathVision,0.257588,0.273866,0.331511,0.236904,0.334122,0.415919,0.457001
TQA,0.759932,0.805365,0.802968,0.690677,0.810674,0.847489,0.858219
ScienceQA,0.829853,0.815754,0.785635,0.726543,0.816404,0.851326,0.854766
MMStar,0.498083,0.534167,0.553708,0.463042,0.593875,0.582333,0.643042
MMMU,NaN,NaN,NaN,NaN,NaN,NaN,NaN


16 16
Processing llama 203025 ...
Processing llama 172050 ...
Processing llama 202757 ...
Processing llama 171853 ...
Processing llama 140700 ...
Processing llama 140627 ...
Processing llama 221548 ...
Processing llama 105714 ...
Processing llama 105502 ...
Processing llama 074713 ...
Processing llama 074346 ...
Processing llama 043727 ...
Processing llama 043150 ...
Processing llama 012539 ...
Processing llama 012003 ...
Processing llama 220740 ...
Processing pixtral 213932 ...
Processing pixtral 214327 ...
Processing pixtral 002652 ...
Processing pixtral 002828 ...
Processing pixtral 031115 ...
Processing pixtral 055750 ...
Processing pixtral 031642 ...
Processing pixtral 060201 ...
Processing pixtral 165810 ...
Processing pixtral 084233 ...
Processing pixtral 084919 ...
Processing pixtral 112750 ...
Processing pixtral 113801 ...
Processing pixtral 141125 ...
Processing pixtral 142426 ...
Processing pixtral 170951 ...
Processing gemma3_12b 064105 ...
Processing gemma3_12b 064147 ...


,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.548843,0.601968,0.683681,0.599769,0.772801,0.822917,0.833218
MathVision,0.257588,0.273866,0.331511,0.236904,0.334122,0.415919,0.457001
TQA,0.759932,0.805365,0.802968,0.690677,0.810674,0.847489,0.858219
ScienceQA,0.829853,0.815754,0.785635,0.726543,0.816404,0.851326,0.854766
MMStar,0.498083,0.534167,0.553708,0.463042,0.593875,0.582333,0.643042
MMMU,0.475466,0.505823,0.550388,0.418866,0.526398,0.606599,0.66778


In [7]:
df_original = pd.read_csv("data/final_results/multi_round/lr_feature_all_model_multi_round_cot_trigger.csv", index_col=0)
display(df_original, df)
# only keep the columns that are in df, 
# Filter df_original to only keep columns that exist in df
df_original_filtered = df_original[df.columns]
mv_delta_all = df - df_original_filtered * 0.01
display(mv_delta_all)
mv_delta_all.to_csv(f"data/final_results/multi_round/mv_delta_all_{MODEL_LIST}.csv", index=True)

,pixtral,gemma3_12b,llama,qwen_3b,qwen_7b,qwen_32b,qwen_72b,avg_acc,pivot_avg,maxlength_avg,voting_avg,feature_all
MathVista,56.030093,65.034722,52.037037,51.944444,72.083333,78.576389,80.578704,65.183532,64.894180,61.798942,69.462632,64.841270
MathVision,25.199902,31.837467,23.413022,22.270725,30.177056,38.797324,42.889197,30.654956,29.839612,27.722865,32.952956,30.548303
TQA,77.342085,78.858447,70.407154,60.852359,78.500761,83.057458,84.524353,76.220374,76.033920,73.581213,79.680637,76.264405
ScienceQA,78.318666,77.832176,77.841472,66.667700,79.759544,84.209222,84.643034,78.467402,78.263333,77.066364,81.110118,78.546639
MMStar,50.350000,53.395833,46.087500,41.225000,56.770833,56.337500,62.558333,52.389286,51.619048,50.161905,55.264286,52.190476
MMMU,47.647516,52.492236,42.872671,37.406832,50.527950,59.037267,64.184783,50.595608,50.505768,49.405501,53.574756,51.268855
average,55.814710,59.908480,52.109809,46.727843,61.303246,66.669193,69.896401,58.918526,58.525977,56.622798,62.007564,58.977422


,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.548843,0.601968,0.683681,0.599769,0.772801,0.822917,0.833218
MathVision,0.257588,0.273866,0.331511,0.236904,0.334122,0.415919,0.457001
TQA,0.759932,0.805365,0.802968,0.690677,0.810674,0.847489,0.858219
ScienceQA,0.829853,0.815754,0.785635,0.726543,0.816404,0.851326,0.854766
MMStar,0.498083,0.534167,0.553708,0.463042,0.593875,0.582333,0.643042
MMMU,0.475466,0.505823,0.550388,0.418866,0.526398,0.606599,0.66778
average,0.561627,0.58949,0.617982,0.522634,0.642379,0.687764,0.719004


,llama,pixtral,gemma3_12b,qwen_3b,qwen_7b,qwen_32b,qwen_72b
MathVista,0.028472,0.041667,0.033333,0.080324,0.051968,0.037153,0.027431
MathVision,0.023458,0.021867,0.013136,0.014197,0.032352,0.027945,0.028109
TQA,0.05586,0.031944,0.014384,0.082154,0.025666,0.016914,0.012976
ScienceQA,0.051438,0.032567,0.007313,0.059866,0.018809,0.009234,0.008335
MMStar,0.037208,0.030667,0.01975,0.050792,0.026167,0.018958,0.017458
MMMU,0.046739,0.029348,0.025466,0.044798,0.021118,0.016227,0.025932
average,0.040529,0.031343,0.018897,0.055355,0.029346,0.021072,0.02004


In [9]:
print("NMI for each Dataset")
for DATASET_NAME in DATASET_LIST:
    print(DATASET_NAME)
    for MODEL_NAME in MODEL_LIST:
        print(f"{nmi_all[MODEL_NAME][DATASET_NAME]} {mv_delta_all.loc[DATASET_NAME][MODEL_NAME]}")

NMI for each Dataset
MathVista
0.12506253773210915 0.028472222222222232
0.1407469971730799 0.04166666666666652
0.22121046296895408 0.03333333333333344
0.06600692889671005 0.08032407407407405
0.17772193818350726 0.05196759259259254
0.3109096605248164 0.03715277777777781
0.29960633223076366 0.027430555555555514
MathVision
0.02259082046024646 0.023457898172323854
0.039855736721699925 0.021866840731070536
0.1343662345124931 0.013136422976501305
0.008241235443607615 0.014197127937336879
0.06336350628932781 0.03235150130548303
0.1497352914466856 0.027945496083550958
0.17419624459518446 0.028108681462141072
TQA
0.15879199656075815 0.05585996955859973
0.27400923033362734 0.03194444444444455
0.4151015698496539 0.014383561643835363
0.08178600738319038 0.08215372907153728
0.279719349269899 0.02566590563165916
0.39980238891141406 0.016914003044140036
0.41105837909282467 0.012975646879756386
ScienceQA
0.15466061033373482 0.051437778879524054
0.22921294389732752 0.03256693108577091
0.437105187326588

In [11]:
print("Rho for each Dataset")
for DATASET_NAME in DATASET_LIST:
    print(DATASET_NAME)
    for MODEL_NAME in MODEL_LIST:
        print(f"{rho_all[MODEL_NAME][DATASET_NAME]} {mv_delta_all.loc[DATASET_NAME][MODEL_NAME]}")

Rho for each Dataset
MathVista
0.5449761036644336 0.028472222222222232
0.5546185741946377 0.04166666666666652
0.6811205660842521 0.03333333333333344
0.4787017305897051 0.08032407407407405
0.7198648121890151 0.05196759259259254
0.8067338894054553 0.03715277777777781
0.7992185067827278 0.027430555555555514
MathVision
0.6069284023788312 0.023457898172323854
0.6559951638675384 0.021866840731070536
0.6601019758562058 0.013136422976501305
0.566734702522583 0.014197127937336879
0.631474691236384 0.03235150130548303
0.6551299674220367 0.027945496083550958
0.7153838682998538 0.028108681462141072
TQA
0.6949457797515505 0.05585996955859973
0.8009648475881782 0.03194444444444455
0.8631955443618481 0.014383561643835363
0.5705405383768252 0.08215372907153728
0.8082136969870573 0.02566590563165916
0.8786113800867235 0.016914003044140036
0.8861695002475114 0.012975646879756386
ScienceQA
0.6896679246607068 0.051437778879524054
0.7481229693538438 0.03256693108577091
0.8459557484151924 0.0073128408527515

In [65]:
for MODEL_NAME in MODEL_LIST:
    nmi_all[MODEL_NAME]['average'] = np.mean(list(nmi_all[MODEL_NAME].values()))
    print(f"{nmi_all[MODEL_NAME]['average']} {mv_delta_all.loc['average'][MODEL_NAME]}")

0.10865103143693278 0.04052922210013088
0.16694106474118298 0.03134322928026256
0.29607655733403315 0.018896999552622917
0.058333002760592056 0.05535514537067254
0.20269462097893953 0.02934642553082467
0.314257221899171 0.021071888203437306
0.33757829526989575 0.020040048892834172


In [66]:
for MODEL_NAME in MODEL_LIST:
    rho_all[MODEL_NAME]['average'] = np.mean(list(rho_all[MODEL_NAME].values()))
    print(f"{rho_all[MODEL_NAME]['average']} {mv_delta_all.loc['average'][MODEL_NAME]}")

0.6006349105923761 0.04052922210013088
0.6593337999567225 0.03134322928026256
0.7485089153551084 0.018896999552622917
0.5113431052530255 0.05535514537067254
0.7145978117844887 0.02934642553082467
0.7827027293666758 0.021071888203437306
0.808790397980558 0.020040048892834172


In [ ]:
type(nmi_qwen7b['MathVista'])

str

In [ ]:
corr_over_rounds = []
for n in range(2, N+1):
    corr_list = []
    for i in range(n):
        for j in range(i+1, n):
            rho_qwen7b = np.corrcoef(multi_match_list_all[0][i], multi_match_list_all[0][j])
            corr_list.append(rho_qwen7b)
    corr_over_rounds.append(np.mean(corr_list))
for i in range(2, 16+1):
    print(f"            ({i}, {corr_over_rounds[i-2]:.2f})")

            (2, 0.77)
            (3, 0.76)
            (4, 0.77)
            (5, 0.76)
            (6, 0.77)
            (7, 0.77)
            (8, 0.77)
            (9, 0.76)
            (10, 0.77)
            (11, 0.77)
            (12, 0.77)
            (13, 0.77)
            (14, 0.77)
            (15, 0.77)
            (16, 0.77)


## NMI

In [ ]:
from sklearn.metrics import normalized_mutual_info_score
nmi_over_rounds = []
for n in range(2, N+1):
    nmi_list = []
    for i in range(n):
        for j in range(i+1, n):
            nmi_qwen7b = normalized_mutual_info_score(multi_match_list_all[0][i], multi_match_list_all[0][j], average_method='min')
            nmi_list.append(nmi_qwen7b)
    nmi_over_rounds.append(np.mean(nmi_list))
for i in range(2, 16+1):
    print(f"            ({i}, {nmi_over_rounds[i-2]:.2f})")

            (2, 0.22)
            (3, 0.21)
            (4, 0.22)
            (5, 0.22)
            (6, 0.22)
            (7, 0.22)
            (8, 0.22)
            (9, 0.21)
            (10, 0.22)
            (11, 0.22)
            (12, 0.22)
            (13, 0.22)
            (14, 0.22)
            (15, 0.22)
            (16, 0.22)


: 

## Build correctness dict

In [ ]:
# model_pairs = [(x1, x2) for i, x1 in enumerate(MODEL_LIST) for x2 in MODEL_LIST[i:] if x1 != x2]
# model_pairs

[('pixtral', 'gemma3_12b'), ('pixtral', 'llama'), ('gemma3_12b', 'llama')]

In [182]:
model_correctness_data = {
    model: multi_match_list_all[i][0].copy() for i, model in enumerate(MODEL_LIST)
}
len(model_correctness_data[MODEL_LIST[0]])

1532

In [45]:
import random

# #shuffle the correctness data for each model
random_correctness_data = {
    model: multi_match_list_all[i][0].copy() for i, model in enumerate(MODEL_LIST)
}
for model in MODEL_LIST:
    random.shuffle(random_correctness_data[model])

In [51]:
# Calculate a baseline for kappa score which 4 arrays are 95% 1 and 5% 0 and shuffle them to four different arrays
baseline_correctness = [1] * 1425 + [0] * 75
baseline_correctness = np.array(baseline_correctness)
# Shuffle baseline_correctness to create 4 different arrays
np.random.seed(42)  # For reproducibility
base_1 = baseline_correctness.copy()
np.random.shuffle(base_1)

np.random.seed(43)
base_2 = baseline_correctness.copy() 
np.random.shuffle(base_2)

np.random.seed(44)
base_3 = baseline_correctness.copy()
np.random.shuffle(base_3)

np.random.seed(45)
base_4 = baseline_correctness.copy()
np.random.shuffle(base_4)

base_correctness_data = {
    model: [base_1, base_2, base_3, base_4][i] for i, model in enumerate(MODEL_LIST)
}

In [183]:
round_correctness_data = {
    # use the first model data to create the round correctness data, with N=16 rounds
    # multi_match_list_all[0][:N][:]
    f"round {i}": multi_match_list_all[0][i][:] for i in range(N)
}

## Fleish Kappa

### Separately

In [17]:
from sklearn.metrics import cohen_kappa_score
def calculate_cohen_kappa(model1, model2, correctness_data, verbose=False):
    """
    Calculate Cohen's Kappa score between two models.
    """
    same_answers = sum(1 for i in range(len(correctness_data[model1])) if correctness_data[model1][i] == correctness_data[model2][i])
    different_answers = len(correctness_data[model1]) - same_answers
    if verbose:
        print(f"{model1} and {model2} have {same_answers} same answers and {different_answers} different answers.")
    kappa = cohen_kappa_score(correctness_data[model1], correctness_data[model2])
    return kappa, same_answers, different_answers
print("Cohen's Kappa Scores between model pairs:")
print("fleish kappa scores interpretation: 0.0-0.2: no agreement, 0.2-0.4: slight agreement, 0.4-0.6: moderate agreement, 0.6-0.8: substantial agreement, 0.8-1.0: almost perfect agreement")
for model1, model2 in model_pairs:
    kappa, same_answers, different_answers = calculate_cohen_kappa(model1, model2, model_correctness_data, verbose=True)
    print(f"{model1} vs {model2}: κ = {kappa:.4f}")

Cohen's Kappa Scores between model pairs:
fleish kappa scores interpretation: 0.0-0.2: no agreement, 0.2-0.4: slight agreement, 0.4-0.6: moderate agreement, 0.6-0.8: substantial agreement, 0.8-1.0: almost perfect agreement


NameError: name 'model_pairs' is not defined

In [47]:
print("Cohen's Kappa Scores for randomly shuffled data between model pairs:")
for model1, model2 in model_pairs:
    kappa, same_answers, different_answers = calculate_cohen_kappa(model1, model2, random_correctness_data, verbose=True)
    print(f"{model1} vs {model2}: κ = {kappa:.4f}")

Cohen's Kappa Scores for randomly shuffled data between model pairs:
qwen_3b and qwen_7b have 274 same answers and 266 different answers.
qwen_3b vs qwen_7b: κ = -0.0286
qwen_3b and qwen_32b have 279 same answers and 261 different answers.
qwen_3b vs qwen_32b: κ = -0.0213
qwen_3b and qwen_72b have 291 same answers and 249 different answers.
qwen_3b vs qwen_72b: κ = 0.0185
qwen_7b and qwen_32b have 351 same answers and 189 different answers.
qwen_7b vs qwen_32b: κ = 0.0499
qwen_7b and qwen_72b have 355 same answers and 185 different answers.
qwen_7b vs qwen_72b: κ = 0.0254
qwen_32b and qwen_72b have 380 same answers and 160 different answers.
qwen_32b vs qwen_72b: κ = 0.0494


In [ ]:
# for i in range(N):
#     for j in range(i+1, N):
#         model1 = f"round {i}"
#         model2 = f"round {j}"
#         kappa, same_answers, different_answers = calculate_cohen_kappa(model1, model2, round_correctness_data, verbose=True)
#         print(f"{model1} vs {model2}: κ = {kappa:.4f}")

round 0 and round 1 have 349 same answers and 191 different answers.
round 0 vs round 1: κ = 0.2897
round 0 and round 2 have 339 same answers and 201 different answers.
round 0 vs round 2: κ = 0.2540
round 0 and round 3 have 359 same answers and 181 different answers.
round 0 vs round 3: κ = 0.3250
round 0 and round 4 have 341 same answers and 199 different answers.
round 0 vs round 4: κ = 0.2609
round 0 and round 5 have 350 same answers and 190 different answers.
round 0 vs round 5: κ = 0.2927
round 0 and round 6 have 355 same answers and 185 different answers.
round 0 vs round 6: κ = 0.3143
round 0 and round 7 have 356 same answers and 184 different answers.
round 0 vs round 7: κ = 0.3211
round 0 and round 8 have 361 same answers and 179 different answers.
round 0 vs round 8: κ = 0.3334
round 0 and round 9 have 350 same answers and 190 different answers.
round 0 vs round 9: κ = 0.2941
round 0 and round 10 have 361 same answers and 179 different answers.
round 0 vs round 10: κ = 0.333

### Together

In [184]:
from statsmodels.stats.inter_rater import fleiss_kappa

# Create a matrix for Fleiss' kappa calculation
# Each row represents one question, each column represents agreement counts
# For binary ratings (0=incorrect, 1=correct), we need counts of each rating
def cal_joint_fleiss_kappa(model_correctness_data):
    n_questions = len(ds)
    n_raters = len(model_correctness_data)  # 4 models

    # Create the rating matrix
    ratings_matrix = np.zeros((n_questions, 2))  # 2 categories: 0 (incorrect) and 1 (correct)

    for i in range(n_questions):
        # Count how many models got question i correct (1) and incorrect (0)
        correct_count = sum(1 for model in model_correctness_data.keys() if model_correctness_data[model][i] == 1)
        
        incorrect_count = n_raters - correct_count
        
        ratings_matrix[i, 0] = incorrect_count  # count of 0s (incorrect)
        ratings_matrix[i, 1] = correct_count    # count of 1s (correct)

    # Calculate Fleiss' kappa
    kappa_fleiss = fleiss_kappa(ratings_matrix, method='fleiss')
    return kappa_fleiss

kappa_fleiss = cal_joint_fleiss_kappa(model_correctness_data)
print(f"Fleiss' Kappa: κ = {kappa_fleiss:.4f}")
# Note: fleiss_kappa function does not return p-value directly

Fleiss' Kappa: κ = 0.1225


In [81]:
kappa_fleiss = cal_joint_fleiss_kappa(random_correctness_data)
print(f"Fleiss' Kappa: κ = {kappa_fleiss:.4f}")

Fleiss' Kappa: κ = -0.0077


In [80]:
kappa_fleiss = cal_joint_fleiss_kappa(base_correctness_data)
print(f"Fleiss' Kappa for baseline: κ = {kappa_fleiss:.4f}")

Fleiss' Kappa for baseline: κ = 0.0003


In [185]:
kappa_fleiss = cal_joint_fleiss_kappa(round_correctness_data)
print(f"Fleiss' Kappa for rounds: κ = {kappa_fleiss:.4f}")

Fleiss' Kappa for rounds: κ = 0.4630


## Cramer

In [75]:
from scipy.stats.contingency import association
from sklearn.metrics import mutual_info_score

def create_contingency_table(correctness_data, model1, model2, verbose=True):
    contingency_table = pd.crosstab(correctness_data[model1], correctness_data[model2], rownames=[model1], colnames=[model2])
    assoc = association(contingency_table, method='cramer')
    if verbose:
        print(f"Association (Cramer Coefficient) between {model1} and {model2}: {assoc:.4f}")
    return assoc

for pair in model_pairs:
    create_contingency_table(model_correctness_data, pair[0], pair[1])
    # create_contingency_table(random_correctness_data, pair[0], pair[1])
    # create_contingency_table(base_correctness_data, pair[0], pair[1])


Association (Cramer Coefficient) between qwen_3b and qwen_7b: 0.2365
Association (Cramer Coefficient) between qwen_3b and qwen_32b: 0.2395
Association (Cramer Coefficient) between qwen_3b and qwen_72b: 0.1997
Association (Cramer Coefficient) between qwen_7b and qwen_32b: 0.4176
Association (Cramer Coefficient) between qwen_7b and qwen_72b: 0.3337
Association (Cramer Coefficient) between qwen_32b and qwen_72b: 0.4207


In [76]:
assoc_list = []
for i in range(N):
    for j in range(i+1, N):
        model1 = f"round {i}"
        model2 = f"round {j}"
        assoc_list.append(create_contingency_table(round_correctness_data, model1, model2, False))
np.mean(assoc_list)

np.float64(0.2974646800592223)

## PMI (Pointwise mutual information)

In [62]:
def calculate_pmi(model1, model2, correctness_data):
    """
    Calculate Pointwise Mutual Information (PMI) between two models' correctness.
    PMI(X=x, Y=y) = log(P(X=x, Y=y) / (P(X=x) * P(Y=y)))
    """
    from collections import Counter
    import math
    
    # Get correctness arrays for both models
    x = correctness_data[model1]
    y = correctness_data[model2]
    
    n = len(x)
    
    # Calculate joint and marginal probabilities
    joint_counts = Counter(zip(x, y))
    x_counts = Counter(x)
    y_counts = Counter(y)
    
    # Calculate PMI for each combination
    pmi_results = {}
    
    for (x_val, y_val), joint_count in joint_counts.items():
        # Joint probability
        p_xy = joint_count / n
        
        # Marginal probabilities
        p_x = x_counts[x_val] / n
        p_y = y_counts[y_val] / n
        
        # PMI calculation
        if p_x > 0 and p_y > 0 and p_xy > 0:
            pmi = math.log(p_xy / (p_x * p_y))
            pmi_results[(x_val, y_val)] = pmi
        else:
            pmi_results[(x_val, y_val)] = float('-inf')
    
    return pmi_results

def calculate_average_pmi(model1, model2, correctness_data):
    """
    Calculate weighted average PMI between two models.
    """
    pmi_results = calculate_pmi(model1, model2, correctness_data)
    
    # Calculate weights based on joint probabilities
    x = correctness_data[model1]
    y = correctness_data[model2]
    n = len(x)
    joint_counts = Counter(zip(x, y))
    
    weighted_pmi = 0
    total_weight = 0
    
    for (x_val, y_val), pmi in pmi_results.items():
        if pmi != float('-inf'):
            weight = joint_counts[(x_val, y_val)] / n
            weighted_pmi += weight * pmi
            total_weight += weight
    
    if total_weight > 0:
        return weighted_pmi / total_weight
    else:
        return 0

# Calculate PMI for all model pairs
print("Pointwise Mutual Information (PMI) between model pairs:")
print("PMI interpretation: 0 = independence, >0 = positive association, <0 = negative association")
print()

for model1, model2 in model_pairs:
    pmi_results = calculate_pmi(model1, model2, model_correctness_data)
    avg_pmi = calculate_average_pmi(model1, model2, model_correctness_data)
    
    print(f"{model1} vs {model2}:")
    print(f"  Average PMI: {avg_pmi:.4f}")
    print("  PMI for each outcome combination:")
    for (x_val, y_val), pmi in pmi_results.items():
        if pmi != float('-inf'):
            print(f"    PMI({model1}={x_val}, {model2}={y_val}): {pmi:.4f}")
        else:
            print(f"    PMI({model1}={x_val}, {model2}={y_val}): -∞ (zero probability)")
    print()

Pointwise Mutual Information (PMI) between model pairs:
PMI interpretation: 0 = independence, >0 = positive association, <0 = negative association

qwen_3b vs qwen_7b:
  Average PMI: 0.0281
  PMI for each outcome combination:
    PMI(qwen_3b=0, qwen_7b=1): -0.1729
    PMI(qwen_3b=0, qwen_7b=0): 0.3538
    PMI(qwen_3b=1, qwen_7b=1): 0.1238
    PMI(qwen_3b=1, qwen_7b=0): -0.4346

qwen_3b vs qwen_32b:
  Average PMI: 0.0289
  PMI for each outcome combination:
    PMI(qwen_3b=0, qwen_32b=1): -0.1461
    PMI(qwen_3b=1, qwen_32b=1): 0.1070
    PMI(qwen_3b=0, qwen_32b=0): 0.4108
    PMI(qwen_3b=1, qwen_32b=0): -0.5481

qwen_3b vs qwen_72b:
  Average PMI: 0.0200
  PMI for each outcome combination:
    PMI(qwen_3b=0, qwen_72b=1): -0.1060
    PMI(qwen_3b=0, qwen_72b=0): 0.3902
    PMI(qwen_3b=1, qwen_72b=1): 0.0802
    PMI(qwen_3b=1, qwen_72b=0): -0.5047

qwen_7b vs qwen_32b:
  Average PMI: 0.0794
  PMI for each outcome combination:
    PMI(qwen_7b=1, qwen_32b=1): 0.1241
    PMI(qwen_7b=0, qwen_3

In [1]:
from sklearn.metrics import normalized_mutual_info_score